Field Validator

In [1]:
from pydantic import BaseModel, Field, EmailStr, AnyUrl
from typing import List, Dict, Optional, Annotated

In [2]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

In [3]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient1 = Patient(**patient_info)
print(patient1)

name='benky' email='benky@example.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


Let's say you want to validate the email field to validate bank users alone

In [4]:
from pydantic import field_validator

In [5]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value

In [6]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient2 = Patient(**patient_info)
print(patient2)

ValidationError: 1 validation error for Patient
email
  Value error, Email domain must be one of ['sbi.com', 'equitas.com'] [type=value_error, input_value='benky@example.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [7]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient3 = Patient(**patient_info)
print(patient3)

name='benky' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


We can also perform transformations using field validator

In [8]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()

In [9]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient4 = Patient(**patient_info)
print(patient4)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


Mode: After (default) type coercion

Mode: Before type coercion

In [12]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value

Age is sent as a string and field validator uses the string now as the mode is set to before

In [13]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient5 = Patient(**patient_info)
print(patient5)

TypeError: '<' not supported between instances of 'str' and 'int'

In [15]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': 30, 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient6 = Patient(**patient_info)
print(patient6)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


In [16]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value

In [17]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient7 = Patient(**patient_info)
print(patient7)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}
